# 小说有声书流水线：TXT → 分角色章节 WAV

把小说文本做成**多角色有声书**（IndexTTS-2.5 克隆音色 + 情感）。

**会做什么**
- 分析全书风格，抽出旁白和人物表
- 从种子库匹配音色，再用 2.5 为每个角色自制声卡
- 按旁白 / 对白拆成适合 2.5 的朗读句（句长、拼音标注、情感向量）
- 按顺序合成，再按章节合并成 WAV
- 中途断开可从检查点接着跑；某一章读错可只重做那一章

**开始前请准备**
1. **Runtime → 更改运行时类型 → GPU（T4 即可）**
2. 在 [Colab 密钥](https://colab.research.google.com/notebooks/secrets.ipynb) 里添加：
   - `LLM_API_KEY`：DeepSeek / OpenAI 兼容接口的 Key（[platform.deepseek.com](https://platform.deepseek.com)）
3. 准备一段 **3–15 秒** 干净人声 WAV，作为旁白种子，也是缺种子时的全员回退音色


## 0. 挂载 Google 云端硬盘（模型缓存）

大模型（IndexTTS-2.5）会写到
`我的云端硬盘/index-tts-cache/`，下次开 session 不用重下。
请点「连接到 Google 云端硬盘」并允许访问。


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Model cache on Google Drive (avoids re-downloading every session) ---
DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.makedirs(f"{DRIVE_CACHE}/hf_home", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/torch_home", exist_ok=True)

# HuggingFace models: IndexTTS-2.5
# Only set HF_HOME — HF_HUB_CACHE auto-derives as HF_HOME/hub
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

print(f"Model cache: {DRIVE_CACHE}")
print(f"HF_HOME:    {os.environ['HF_HOME']}")
print(f"TORCH_HOME: {os.environ['TORCH_HOME']}")


## 1. 检查 GPU、克隆仓库、安装依赖

安装单元格跑完会**自动重启运行时**。重启后不要重装，只跑下一格「恢复环境」。


In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")


In [ ]:
# 克隆 py3.12 分支（已有仓库则拉取更新）
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git 2>/dev/null || (cd /content/index-tts && git pull)
%cd /content/index-tts


In [ ]:
%cd /content/index-tts
!bash tools/setup_colab.sh --extra novel

print("Restarting runtime to apply package changes...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


### 运行时重启后：先跑这一格恢复环境

上一格会关掉内核。从这里继续：重新挂载硬盘、回到项目目录、确认 torch/cuda 正常。**不要再跑安装格。**


In [ ]:
import os
from google.colab import drive

# Re-mount Drive and restore env vars after runtime restart
drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

%cd /content/index-tts

# Verify packages loaded correctly
import torch, numpy, numba
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
print(f"numpy={numpy.__version__}")
print(f"numba={numba.__version__}")
print("All good!")


## 2. 下载 IndexTTS-2.5 权重

权重大约数 GB，下到云端硬盘的 `index-tts-cache/checkpoints-2.5/`，再软链到项目里的 `checkpoints/`。
已经下过会自动跳过。


In [ ]:
import os

DRIVE_CKPT = f"{DRIVE_CACHE}/checkpoints-2.5"
os.environ["INDEX_TTS_MODEL_DIR"] = DRIVE_CKPT
LOCAL_CKPT = "/content/index-tts/checkpoints"

# Download to Drive (persists across sessions)
if not os.path.exists(f"{DRIVE_CKPT}/config.yaml"):
    print("Downloading IndexTTS-2.5 checkpoints to Google Drive (first time only)...")
    !huggingface-cli download IndexTeam/IndexTTS-2.5 --local-dir "{DRIVE_CKPT}"
else:
    print(f"Checkpoints already cached at {DRIVE_CKPT}")

# Symlink so the pipeline finds them at the expected local path
if os.path.islink(LOCAL_CKPT):
    os.unlink(LOCAL_CKPT)
elif os.path.exists(LOCAL_CKPT):
    import shutil
    shutil.rmtree(LOCAL_CKPT) if os.path.isdir(LOCAL_CKPT) else os.remove(LOCAL_CKPT)
os.symlink(DRIVE_CKPT, LOCAL_CKPT)
print(f"Symlinked: {LOCAL_CKPT} -> {DRIVE_CKPT}")

# Verify
!ls -la checkpoints/config.yaml


## 3. 配置密钥与朗读参数

`LLM_API_KEY` 默认走 **DeepSeek**（`deepseek-v4-flash`）。密钥在 Colab「密钥」里，名称必须是 `LLM_API_KEY`。

| 参数 | 含义 |
|---|---|
| `REF_MODE` | `card`：用 2.5 自制角色声卡；`seed`：直接用种子原声（音质更稳） |
| `LANG` | 覆盖全书语言。`None` 则用风格分析结果（多为 `zh`） |
| `STOP_AFTER` | 只跑到某一步。`characters` 可先看人物表；`None` 跑完全程 |
| `CHAPTER` | 只处理第 N 章（1 起）。`None` 表示全书 |
| `FORCE_TTS` | `True` 删掉目标章已有句子 WAV 再合成 |
| `CONCAT_BOOK` | `True` 额外拼一本 `book.wav`（章间静音 1.5s） |
| `WORK_DIR` | 中间文件目录。默认写云盘，断线可续跑 |
| `CLEANUP` | `False` 才能事后只重做一章。不要和 `CHAPTER` 一起开 `True` |
| `REF_AUDIO` | 旁白种子。默认用第 4 节下载的 `examples/voice_05.wav` |


In [ ]:
from pathlib import Path
from google.colab import userdata

try:
    LLM_API_KEY = userdata.get('LLM_API_KEY')
except Exception:
    LLM_API_KEY = None

# --- 分析 / 拆句模型（默认 DeepSeek；要换就注释/取消注释） ---
LLM_API_BASE = "https://api.deepseek.com/v1"    ; LLM_MODEL = "deepseek-v4-flash"
# LLM_API_BASE = "https://api.openai.com/v1"       ; LLM_MODEL = "gpt-4o-mini"
# LLM_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/" ; LLM_MODEL = "gemini-2.0-flash"

# --- 朗读参数 ---
USE_FP16 = True
REF_MODE = "card"          # "card" 自制声卡；"seed" 直接用种子 wav
LANG = None                # "zh" / "en" / "ja" / "es" / "ar"；None=跟风格分析
STOP_AFTER = None          # "chapters" | "style" | "characters" | "voices" | "script" | "tts" | "merge"
CHAPTER = None             # 只跑第 N 章（1 起）；None=全书
FORCE_TTS = False
STRICT = False             # True=某句 TTS 失败就停；False=写静音继续
CONCAT_BOOK = False
CLEANUP = False            # 必须 False，才能事后只重做一章
WORK_DIR = f"{DRIVE_CACHE}/novel_workspace"  # 默认写云盘，断线可续跑
VOICE_BANK = None          # None=用 examples/voice_bank.yaml（voice_01/04/05）
# 旁白 / 全员回退种子。第 4 节下载后 examples/voice_05.wav 即可用。
REF_AUDIO = "examples/voice_05.wav"
# REF_AUDIO = "/content/drive/MyDrive/voices/narrator.wav"  # 改用自己的音色

print(f"LLM: {LLM_MODEL} @ {LLM_API_BASE}")
print(f"ref_mode={REF_MODE} lang={LANG} stop_after={STOP_AFTER} chapter={CHAPTER}")
print(f"WORK_DIR={WORK_DIR}")
print(f"REF_AUDIO={REF_AUDIO}")


## 4. 预下载示例种子音色（可选）

仓库里的 `examples/voice_*.wav` 默认不入库。跑这一格会把官方示例音色拉到 `examples/`。

- `voice_05.wav`：默认旁白种子（第 3 节 `REF_AUDIO`）
- `voice_01` / `04` / `05`：`examples/voice_bank.yaml` 按性别/年龄给角色配种
- 下载完成后**请再跑一次第 3 节**，让 `REF_AUDIO` 指到这些文件

不想用示例音色时，把 `REF_AUDIO` 改成你自己的云盘 WAV。


In [ ]:
# Pre-download official example voices into examples/ (optional)
from indextts.utils.examples_downloader import ensure_examples_available

print("Downloading example voice WAVs if missing...")
ensure_examples_available()
!ls -la examples/*.wav 2>/dev/null | head
print("Example voices ready (or already cached).")


## 5. 处理一本小说

已有 `novel_path` / `ref_audio` 且文件还在时直接续跑，不会再弹上传。换书把 `FORCE_UPLOAD = True`，或改用下面的 Drive 路径。

**输入**
- 小说：UTF-8 的 `.txt` / `.md`
- 参考音频：3–15 秒单人干声 WAV（旁白 + 缺种子时的回退）

同一本书中途失败，用相同 `novel_path` 和 `WORK_DIR` 再跑，会从检查点接着做。


In [ ]:
from novel_pipeline import run_novel_pipeline
from google.colab import files
from pathlib import Path
import os

# 续跑：保留已有 novel_path / ref_audio。换书：FORCE_UPLOAD = True，或改用下面的 Drive 路径。
FORCE_UPLOAD = False
# novel_path = "/content/drive/MyDrive/novels/book.txt"
# ref_audio = "/content/drive/MyDrive/voices/narrator.wav"

if FORCE_UPLOAD or "novel_path" not in dir() or not Path(str(novel_path)).is_file():
    print("Upload the novel (.txt / .md):")
    uploaded = files.upload()
    novel_path = list(uploaded.keys())[0]

if FORCE_UPLOAD or "ref_audio" not in dir() or not Path(str(ref_audio)).is_file():
    print("Upload narrator / fallback reference audio (3-15s WAV):")
    uploaded_ref = files.upload()
    ref_audio = list(uploaded_ref.keys())[0]

# === 开始朗读（中断后再跑会自动续）===
stem = Path(novel_path).stem
os.makedirs(f"{DRIVE_CACHE}/output", exist_ok=True)
output_dir = f"{DRIVE_CACHE}/output/{stem}_chapters"

result = run_novel_pipeline(
    input_path=novel_path,
    output=output_dir,
    work_dir=WORK_DIR,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_dir="checkpoints",
    use_fp16=USE_FP16,
    ref_audio=ref_audio,
    voice_bank=VOICE_BANK,
    ref_mode=REF_MODE,
    lang=LANG,
    stop_after=STOP_AFTER,
    chapter=CHAPTER,
    force_tts=FORCE_TTS,
    strict=STRICT,
    concat_book=CONCAT_BOOK,
    cleanup=CLEANUP,
)

print(f"\nDone. step={result.get('step')} output={result.get('output')}")

from IPython.display import Audio, display
out = result.get("output")
if out and Path(str(out)).is_dir():
    wavs = sorted(Path(out).glob("*.wav"))
    print(f"Chapter WAVs ({len(wavs)}):")
    for w in wavs:
        print(f"  {w}")
    if wavs:
        display(Audio(str(wavs[0])))
elif out and Path(str(out)).is_file():
    display(Audio(str(out)))


In [ ]:
# 下载本章 / 全书 WAV 到本地
from pathlib import Path

out = result.get("output")
if out and Path(str(out)).is_dir():
    for w in sorted(Path(out).glob("*.wav")):
        files.download(str(w))
elif out and Path(str(out)).is_file():
    files.download(str(out))
else:
    print("还没有可下载的成品。先跑完第 5 节。")


## 6. 批量处理云盘文件夹

把待朗读的 `.txt` / `.md` 放进 `INPUT_DIR`，章节 WAV 写到 `OUTPUT_DIR/{书名}_chapters/`。
每本书单独工作目录；已有输出目录且非空时跳过。
参考音频默认用第 3 节的 `REF_AUDIO`（云盘路径）。若第 5 节已经上传过 wav，会优先用那条。


In [ ]:
import os
from novel_pipeline import run_novel_pipeline
from pathlib import Path

INPUT_DIR  = "/content/drive/MyDrive/novels/input"    # 待朗读文本
OUTPUT_DIR = "/content/drive/MyDrive/novels/output"   # 章节 WAV

# 参考音频查找顺序：第 5 节上传 → 第 3 节 REF_AUDIO → 第 4 节示例音色
_ref_candidates = []
if "ref_audio" in dir():
    _ref_candidates.append(ref_audio)
if "REF_AUDIO" in dir():
    _ref_candidates.append(REF_AUDIO)
_ref_candidates.extend([
    "examples/voice_05.wav",
    "examples/voice_01.wav",
    "/content/drive/MyDrive/voices/narrator.wav",
])
BATCH_REF = next((p for p in _ref_candidates if p and Path(str(p)).is_file()), None)
if not BATCH_REF:
    raise FileNotFoundError(
        "找不到参考音频。请先跑第 4 节下载示例音色，"
        "或把旁白 WAV 放到 examples/voice_05.wav / REF_AUDIO。"
    )
print(f"BATCH_REF={BATCH_REF}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
novels = sorted(
    p for p in Path(INPUT_DIR).glob("*")
    if p.suffix.lower() in {".txt", ".md"}
)
print(f"Found {len(novels)} novel(s) in {INPUT_DIR}")

for src in novels:
    out_dir = Path(OUTPUT_DIR) / f"{src.stem}_chapters"
    if out_dir.is_dir() and any(out_dir.glob("*.wav")):
        print(f"Skip (already has WAV): {src.name}")
        continue
    print(f"\n=== {src.name} ===")
    try:
        run_novel_pipeline(
            input_path=str(src),
            output=str(out_dir),
            work_dir=WORK_DIR,
            llm_api_key=LLM_API_KEY,
            llm_api_base=LLM_API_BASE,
            llm_model=LLM_MODEL,
            model_dir="checkpoints",
            use_fp16=USE_FP16,
            ref_audio=BATCH_REF,
            voice_bank=VOICE_BANK,
            ref_mode=REF_MODE,
            lang=LANG,
            stop_after=STOP_AFTER,
            chapter=None,
            force_tts=False,
            strict=STRICT,
            concat_book=CONCAT_BOOK,
            cleanup=CLEANUP,
        )
    except Exception as exc:
        print(f"FAILED {src.name}: {exc}")

print(f"\n全部完成。成品目录: {OUTPUT_DIR}")


---
## 7. 只重做某一章

整本跑完后，若某一章读错，不必重跑分析。设 `CHAPTER` 为章节号（1 起），
`FORCE_TTS = True` 会删掉该章已有句子 WAV 再合成。

需要 `WORK_DIR` 里还有该书的 `checkpoint.json`（所以 `CLEANUP` 必须是 `False`）。
不要和 `CLEANUP=True` 一起用。

发音可用 IndexTTS-2.5 标注（先改 `script/cXX.json` 里的 `tts_text`，再重做该章）：

- 拼音（须在 `checkpoints/pinyin.vocab`）：`他在银<行|HANG2>办理业务`
- 英文 CMU 音素：`我们用<ChatGPT|CH AE1 T JH IY1 P IY1 T IY1>`


In [ ]:
from novel_pipeline import run_novel_pipeline
from pathlib import Path

# 改成要重做的章号（1 起）
REDO_CHAPTER = 1
REDO_NOVEL = novel_path if "novel_path" in dir() else "/content/drive/MyDrive/novels/input/book.txt"
REDO_REF = next(
    (
        p for p in (
            ref_audio if "ref_audio" in dir() else None,
            REF_AUDIO if "REF_AUDIO" in dir() else None,
            "examples/voice_05.wav",
            "examples/voice_01.wav",
        )
        if p and Path(str(p)).is_file()
    ),
    "examples/voice_05.wav",
)
REDO_OUT = output_dir if "output_dir" in dir() else f"{DRIVE_CACHE}/output/{Path(REDO_NOVEL).stem}_chapters"

redone = run_novel_pipeline(
    input_path=REDO_NOVEL,
    output=REDO_OUT,
    work_dir=WORK_DIR,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_dir="checkpoints",
    use_fp16=USE_FP16,
    ref_audio=REDO_REF,
    voice_bank=VOICE_BANK,
    ref_mode=REF_MODE,
    lang=LANG,
    chapter=REDO_CHAPTER,
    force_tts=True,
    strict=STRICT,
    concat_book=False,
    cleanup=False,
)

from IPython.display import Audio, display
stem = Path(REDO_NOVEL).stem
cid = f"c{REDO_CHAPTER:02d}"
redo_wav = Path(WORK_DIR) / stem / "chapters" / f"{cid}.wav"
exported = Path(REDO_OUT) / f"{stem}_{cid}.wav"
play = exported if exported.is_file() else redo_wav
print(f"Rebuilt {cid}: {play}")
if play.is_file():
    display(Audio(str(play)))


## 8. 查看中间结果

中间文件在 `WORK_DIR/书名/`（默认云盘 `index-tts-cache/novel_workspace/`）：
风格、人物表、角色声卡、各章脚本和句子 WAV。

专名读音可改仓库根目录或该书工作目录下的 `pronunciation.yaml`。


In [ ]:
import json, os
from IPython.display import Audio, display

# 打开最近一次朗读的工作目录
work_dirs = sorted(
    [d for d in os.listdir(WORK_DIR) if os.path.isdir(f"{WORK_DIR}/{d}")]
) if os.path.isdir(WORK_DIR) else []

if work_dirs:
    work_dir = f"{WORK_DIR}/{work_dirs[-1]}"
    print(f"Work directory: {work_dir}")
    print(f"Contents: {os.listdir(work_dir)}")

    style_path = f"{work_dir}/style.json"
    if os.path.exists(style_path):
        with open(style_path, encoding="utf-8") as f:
            style = json.load(f)
        print("\n--- 风格 ---")
        for key in ("title", "genre", "tone", "pacing", "lang", "narrator_style"):
            if key in style:
                print(f"  {key}: {style[key]}")

    chars_path = f"{work_dir}/characters.json"
    if os.path.exists(chars_path):
        with open(chars_path, encoding="utf-8") as f:
            chars = json.load(f)
        print(f"\n--- 人物 ({len(chars)}) ---")
        for c in chars[:20]:
            print(
                f"  {c.get('id')}  {c.get('name')}  "
                f"role={c.get('role')} gender={c.get('gender')} "
                f"seed={c.get('seed_voice_id')}"
            )
        if len(chars) > 20:
            print(f"  ... and {len(chars)-20} more")

    voices_dir = f"{work_dir}/voices"
    if os.path.isdir(voices_dir):
        print("\n--- 角色声卡 ---")
        for name in sorted(os.listdir(voices_dir)):
            if name.endswith(".wav"):
                path = f"{voices_dir}/{name}"
                print(f"  {name}")
                display(Audio(path))

    chapters_dir = f"{work_dir}/chapters"
    if os.path.isdir(chapters_dir):
        wavs = sorted(p for p in os.listdir(chapters_dir) if p.endswith(".wav"))
        print(f"\n--- 已合并章节 ({len(wavs)}) ---")
        for name in wavs:
            print(f"  {name}")
        if wavs:
            display(Audio(f"{chapters_dir}/{wavs[0]}"))

    script_dir = f"{work_dir}/script"
    if os.path.isdir(script_dir):
        first = sorted(p for p in os.listdir(script_dir) if p.endswith(".json"))
        if first:
            with open(f"{script_dir}/{first[0]}", encoding="utf-8") as f:
                utts = json.load(f)
            print(f"\n--- 脚本 {first[0]} ({len(utts)} 句) ---")
            for u in utts[:8]:
                print(
                    f"  #{u.get('seq')} [{u.get('speaker_id')}/{u.get('kind')}] "
                    f"{(u.get('tts_text') or u.get('text') or '')[:60]}"
                )
            if len(utts) > 8:
                print(f"  ... and {len(utts)-8} more")
else:
    print("还没有工作目录。请先跑第 5 节朗读一本小说。")
